# Track 2 · Intermediate: 12 stocks with a budget-keeping QAOA

**Qiskit Fall Fest 2026 · IIIT-Delhi · Track 2 (Quantum Optimization)**
Team: _<team name / members>_ · Level: **Intermediate** (builds on our Beginner notebook)

**Task.** Choose exactly **K = 6 of n = 12 stocks** (924 valid portfolios out of 4,096 bitstrings), pick one
improvement route and justify it, and compare against classical methods.

**Our route: (a) a better QAOA.** Specifically, a **constraint-preserving XY mixer** started from a **Dicke state**,
so every circuit run returns a portfolio with exactly 6 stocks, plus layer-by-layer angle initialization.

| Route option (README) | Verdict | Why |
|---|---|---|
| Warm start from a classical relaxation | Not chosen as the main route | At 12 stocks the classical greedy answer is already optimal, so warm-starting from it would hide whether QAOA itself works |
| **Mixer that always keeps exactly K stocks** | **Chosen** | Fixes the weakness our Beginner notebook measured: the penalty λ had to be hand-tuned, and a large λ destroyed answer quality |
| Smarter starting angles | Included | Two warm starts from the previous depth (INTERP interpolation and last-layer repetition), plus random starts and a gradient-based optimizer |
| Quantum machine learning | Not chosen | It needs training data of good portfolios, which means solving the problem first; harder to justify at this size |

> **📘 Study boxes.** Sections marked 📘 explain a concept in plain language. They are meant for learning and
> presenting, and can be skipped when you only want the results.

## 0. Setup

All random seeds are fixed, so a rerun in the **same environment** reproduces every number. Exact values can
shift slightly between library versions (the optimizer is sensitive to floating-point details), so the install
cell pins the Qiskit versions we used and the next cell prints every version.

Our run (Google Colab): Python 3.13, qiskit 2.5.2, qiskit-aer 0.17.2, qiskit-ibm-runtime 0.50.0, numpy 2.1.3,
scipy 1.16.3, pandas 2.2.3, matplotlib 3.10.0.

In [ ]:
# Install packages on Colab if missing (works in plain Python too). Qiskit versions are pinned to our run.
import importlib.util, subprocess, sys
_needed = {"qiskit": "qiskit==2.5.2", "qiskit_aer": "qiskit-aer==0.17.2",
           "qiskit_ibm_runtime": "qiskit-ibm-runtime==0.50.0",
           "scipy": "scipy", "pandas": "pandas", "matplotlib": "matplotlib"}
_missing = [pkg for mod, pkg in _needed.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("Installing:", ", ".join(_missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)

In [ ]:
import sys, time, warnings
from math import comb
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy.optimize import minimize

import qiskit, qiskit_aer, qiskit_ibm_runtime
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import RYGate, XXPlusYYGate
from qiskit.quantum_info import Statevector
from qiskit.primitives import StatevectorSampler
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", module="scipy")

BLUE, ORANGE, AQUA, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c9c8c3", "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "lines.markersize": 8,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 11,
})
print(f"Python {sys.version.split()[0]} | qiskit {qiskit.__version__} | qiskit-aer {qiskit_aer.__version__} | "
      f"qiskit-ibm-runtime {qiskit_ibm_runtime.__version__} | numpy {np.__version__} | scipy {scipy.__version__} | "
      f"pandas {pd.__version__} | matplotlib {matplotlib.__version__}")
if (qiskit.__version__, qiskit_aer.__version__, qiskit_ibm_runtime.__version__) != ("2.5.2", "0.17.2", "0.50.0"):
    print("Note: Qiskit versions differ from our run, so some numbers may differ slightly.")

## 1. The 12-stock problem and its exact answer

Same generator and cost as the README, now with `n, K = 12, 6`:

$$C(x) = q\, x^{\top} \Sigma\, x \;-\; \mu^{\top} x \;+\; \lambda \Big( \sum_i x_i - K \Big)^2$$

> **📘 Why brute force still works.** The number of valid portfolios is $\binom{12}{6} = 924$, so a laptop checks
> them all in milliseconds. That gives us the true answer to grade every method against. It also shows how this
> problem would explode: $\binom{50}{25} \approx 1.3 \times 10^{14}$.

**Bit convention.** Bitstring index $k$ maps to $x_i$ = bit $i$ of $k$, which is Qiskit's qubit order
(qubit 0 is the rightmost character of a measured string).

In [ ]:
n, K, q = 12, 6, 0.5              # stocks, pick exactly K, risk aversion
N = 2 ** n
BITS = ((np.arange(N)[:, None] >> np.arange(n)) & 1).astype(float)   # BITS[k, i] = x_i for bitstring k
WEIGHT = BITS.sum(1).astype(int)                                     # number of stocks bought
FEAS = np.flatnonzero(WEIGHT == K)                                   # the 924 valid portfolios

def make_data(seed):
    """README generator: expected returns mu and covariance sigma for one random dataset."""
    r = np.random.default_rng(seed)
    mu_ = r.uniform(0.05, 0.25, n)
    A_ = r.normal(0, 0.1, (n, n))
    return mu_, A_ @ A_.T

def cost_for(mu_, sigma_, lam=0.0):
    """C(x) for all 4,096 bitstrings of one dataset."""
    return q * np.einsum("ki,ij,kj->k", BITS, sigma_, BITS) - BITS @ mu_ + lam * (WEIGHT - K) ** 2

SEED = 2026
mu, sigma = make_data(SEED)       # the main dataset (same numbers as the README generator)

def cost_table(lam=0.0):
    """C(x) for all 4,096 bitstrings of the main dataset."""
    return cost_for(mu, sigma, lam)

def stocks(k):
    return tuple(i for i in range(n) if (k >> i) & 1)

C0 = cost_table(0.0)                         # cost without penalty
order = FEAS[np.argsort(C0[FEAS])]           # valid portfolios, best first
BEST = order[0]
C_MIN_F, C_MAX_F = C0[FEAS].min(), C0[FEAS].max()
R_RANDOM = (C_MAX_F - C0[FEAS].mean()) / (C_MAX_F - C_MIN_F)   # r of a uniformly random valid portfolio

print(f"{N} bitstrings, {len(FEAS)} valid (= C(12,6) = {comb(12, 6)})")
print(f"Best portfolio: stocks {stocks(BEST)}, cost {C0[BEST]:.4f}")
print(f"Gap to the 2nd-best portfolio: {C0[order[1]] - C0[BEST]:.4f}; a random valid portfolio scores r = {R_RANDOM:.3f}")
pd.DataFrame([{"rank": r + 1, "stocks bought": stocks(k),
               "expected return": round(float(BITS[k] @ mu), 4),
               "risk q·xᵀΣx": round(float(q * BITS[k] @ sigma @ BITS[k]), 4),
               "cost C(x)": round(float(C0[k]), 4)} for r, k in enumerate(order[:5])]).set_index("rank")

### The penalty threshold λ*, in closed form

In the Beginner notebook we found λ* by bisection. There is a direct formula. An invalid portfolio $x$ beats
the best valid one $x^*$ when $C_0(x) + \lambda d^2 < C_0(x^*)$, where $d = \sum x_i - K$. So the penalty must exceed

$$\lambda^* = \max_{x \text{ invalid}} \frac{C_0(x^*) - C_0(x)}{(\sum_i x_i - K)^2}$$

> **📘 Reading the formula.** The numerator is how much cost an invalid portfolio *saves* by breaking the
> budget; the denominator is how hard it gets fined. λ* is set by the "worst offender", the rule-breaker
> with the best saving per unit of fine.

>
> **📘 Two practical notes.** (1) Computing λ* this way needs the brute-force answer, so it is only possible
> at small sizes; we use it here to give the plain-QAOA baseline a fair penalty. (2) At λ = λ* exactly, the
> worst offender *ties* the optimum, so the penalized problem has two best answers. Every penalty we actually
> use is at least **1.05 λ\***.

In [ ]:
def lambda_star(C0_, best_):
    """Smallest λ at which no invalid bitstring beats the best valid one (needs the brute-force answer)."""
    invalid = np.flatnonzero(WEIGHT != K)
    ratios = (C0_[best_] - C0_[invalid]) / (WEIGHT[invalid] - K) ** 2
    return ratios.max(), invalid[ratios.argmax()]

LAMBDA_STAR, offender = lambda_star(C0, BEST)
tie = C0[offender] + LAMBDA_STAR * (WEIGHT[offender] - K) ** 2 - C0[BEST]
print(f"λ* = {LAMBDA_STAR:.4f}; worst offender buys {WEIGHT[offender]} stocks {stocks(offender)}")
print(f"At λ = λ* exactly, the offender ties the optimum (cost difference {tie:.1e}), "
      f"so every penalty we use is at least 1.05 λ*.")

## 2. Why the penalty approach struggles, and the fix

Our Beginner results showed three problems with the penalty encoding:
1. **λ needs tuning.** Below λ* the budget breaks; far above it, answer quality collapses to random.
2. **Wasted samples.** The standard X mixer explores all 4,096 bitstrings, but only 924 (22.6%) are valid.
3. **The ratio r is flattered** by heavily-fined invalid bitstrings.

**The fix: never leave the valid set.** If the starting state contains only 6-stock portfolios, and every gate
preserves the number of stocks, then every measurement is a valid portfolio. No penalty is needed (λ = 0).

> **📘 Hamming weight.** The *Hamming weight* of a bitstring is its number of 1s, here the number of stocks
> bought. The budget rule says "weight = 6". A gate *preserves* the weight if it never turns a 6-stock state
> into a 5- or 7-stock state.
>
> **📘 Why the cost layer is safe.** The cost layer only multiplies each bitstring by a phase $e^{-i\gamma C(x)}$.
> It never moves amplitude between bitstrings, so it cannot change the weight.
>
> **📘 Why the standard mixer is not safe.** $R_X$ on one qubit turns $|0\rangle$ into a mix of $|0\rangle$ and
> $|1\rangle$, so it adds or removes a stock.
>
> **📘 The XY ("swap-like") mixer.** The two-qubit XY gate acts only on $|01\rangle$ and $|10\rangle$:
> $$|01\rangle \to \cos\beta\,|01\rangle - i\sin\beta\,|10\rangle$$
> In portfolio terms it *trades* stock i for stock j: sell one, buy the other. $|00\rangle$ and $|11\rangle$ are
> untouched, so the stock count never changes. Formally it is $e^{-i\beta(XX+YY)/2}$, which commutes with the
> total number of 1s.
>
> **📘 Ring layout.** We apply XY gates between neighbours on a ring (0–1, 1–2, …, 11–0). First the even pairs
> (0–1, 2–3, …), then the odd pairs (1–2, 3–4, …, 11–0). Gates inside each group touch different qubits, so they
> can run at the same time. Repeated trades along the ring can move a stock anywhere.

## 3. The starting state: a Dicke state

The plain QAOA starts in $|+\rangle^{\otimes n}$, an equal mix of **all** 4,096 bitstrings. Our version starts
in the **Dicke state** $|D^{12}_6\rangle$, an equal mix of **only the 924 valid portfolios**:

$$|D^n_K\rangle = \frac{1}{\sqrt{\binom{n}{K}}} \sum_{\text{weight}(x) = K} |x\rangle$$

> **📘 Why a superposition and not one portfolio.** It is unbiased: every valid portfolio starts with the same
> chance, so QAOA can raise good ones from everywhere. Section 8 tests what happens if we start from a single
> portfolio instead.
>
> **📘 How it is built.** We use the deterministic construction of Bärtschi & Eidenbenz (2019). Start from
> $|0\dots0\,1\dots1\rangle$ (6 ones), then apply "split & cyclic shift" blocks made of CNOTs and controlled
> $R_Y$ rotations. Each block moves the right amount of probability from one arrangement of the ones to the
> next. The rotation angles $2\arccos\sqrt{\ell/m}$ set exactly those amounts.

In [ ]:
def _dicke_block(qc, blk, ell, m):
    """Gate (1) when ell = 1, gate (2)_ell otherwise; blk[-1] is the rotated qubit."""
    a, c = blk[-ell - 1], blk[-1]
    theta = -2 * np.arccos(np.sqrt(ell / m))
    qc.cx(c, a)
    if ell == 1:
        qc.cry(theta, a, c)
    else:
        qc.append(RYGate(theta).control(2), [a, blk[-ell], c])
    qc.cx(c, a)

def _scs(qc, m, k):
    """Split & cyclic shift SCS_{m,k} on the last k+1 of the first m qubits."""
    blk = list(range(m))[-(k + 1):]
    for ell in range(1, k + 1):
        _dicke_block(qc, blk, ell, m)

def dicke_circuit(n_, k_):
    qc = QuantumCircuit(n_, name=f"Dicke({n_},{k_})")
    for i in range(n_ - k_, n_):
        qc.x(i)                                  # start from |0...0 1...1>
    for m in range(n_, k_, -1):
        _scs(qc, m, k_)
    for m in range(k_, 1, -1):
        _scs(qc, m, m - 1)
    return qc

# Verify: amplitude 1/sqrt(924) on every valid portfolio, 0 elsewhere
dicke_sv = Statevector(dicke_circuit(n, K)).data
target = np.where(WEIGHT == K, 1 / np.sqrt(len(FEAS)), 0.0)
print(f"Dicke circuit: max |amplitude − target| = {np.abs(dicke_sv - target).max():.1e}")
print(f"Probability on valid portfolios = {np.sum(np.abs(dicke_sv[FEAS]) ** 2):.12f}")
assert np.abs(dicke_sv - target).max() < 1e-10

## 4. Cost Hamiltonian and the full circuit

With the XY mixer we never visit invalid portfolios, so we set **λ = 0** and keep only risk and return. The spin
coefficients come from the same QUBO → spin derivation as the Beginner notebook (reproduced below for any λ,
because the plain-QAOA baseline in Section 7 still needs the penalty).

We also **rescale the costs** to the range [0, 1] over the valid portfolios before building the circuit.
The best answer does not change, but the angle γ then has the same scale for any data set.

The plain-QAOA baseline (Section 7) is scaled by the **same factor**, so an angle γ means the same thing in both
versions and both optimizers search the same γ range.

In [ ]:
def spin_coeffs(lam):
    """C(x) = const + Σ a_i Z_i + Σ_{i<j} b_ij Z_i Z_j  (with x_i = (1 - Z_i)/2)."""
    h = np.array([q * sigma[i, i] - mu[i] + lam * (1 - 2 * K) for i in range(n)])
    Q = np.triu(2 * q * sigma + 2 * lam, 1)                 # pair coefficients, i < j
    a = -h / 2 - (Q + Q.T).sum(1) / 4
    b = Q / 4
    const = lam * K ** 2 + h.sum() / 2 + Q.sum() / 4
    return const, a, b

ZS = 1 - 2 * BITS                                           # Z_i = +1 for x_i = 0, −1 for x_i = 1
for lam in [0.0, 0.3]:
    const, a, b = spin_coeffs(lam)
    spin = const + ZS @ a + np.einsum("ki,ij,kj->k", ZS, b, ZS)
    print(f"λ = {lam}: max |spin form − brute force| over 4,096 bitstrings = {np.abs(spin - cost_table(lam)).max():.1e}")

In [ ]:
EDGES_EVEN = [(i, i + 1) for i in range(0, n, 2)]
EDGES_ODD = [(i, (i + 1) % n) for i in range(1, n, 2)]

def qaoa_circuit(mixer, p, lam, cscale, init="dicke"):
    """mixer='xy': Dicke (or basis) start + XY ring mixer.  mixer='x': |+> start + RX mixer (plain QAOA)."""
    const, a, b = spin_coeffs(lam)
    gam, bet = ParameterVector("γ", p), ParameterVector("β", p)
    qc = QuantumCircuit(n)
    if mixer == "xy":
        if init == "dicke":
            qc.compose(dicke_circuit(n, K), inplace=True)
        else:
            for i in range(n - K, n):
                qc.x(i)                         # a single valid portfolio: stocks 6..11
    else:
        qc.h(range(n))
    for l in range(p):
        for i in range(n):                      # cost layer: e^{-iγ C}
            qc.rz(2 * a[i] * cscale * gam[l], i)
        for i in range(n):
            for j in range(i + 1, n):
                qc.rzz(2 * b[i, j] * cscale * gam[l], i, j)
        if mixer == "xy":                       # XY ring mixer, even pairs then odd pairs
            for e in EDGES_EVEN + EDGES_ODD:
                qc.append(XXPlusYYGate(2 * bet[l]), list(e))
        else:
            for i in range(n):
                qc.rx(2 * bet[l], i)
    return qc, gam, bet

def bind(qc, gam, bet, theta):
    p = len(gam)       # theta = [γ_0..γ_{p-1}, β_0..β_{p-1}]
    return qc.assign_parameters({**{gam[l]: theta[l] for l in range(p)},
                                 **{bet[l]: theta[p + l] for l in range(p)}})

SCALE_XY = 1 / (C_MAX_F - C_MIN_F)
demo, g_, b_ = qaoa_circuit("xy", 1, 0.0, SCALE_XY)
ops = demo.count_ops()
print(f"p = 1 circuit: {ops.get('rzz', 0)} RZZ (one per stock pair), {ops.get('xx_plus_yy', 0)} XY gates, "
      f"plus the Dicke preparation; {demo.num_parameters} parameters")

## 5. Fast training simulator, checked against the Qiskit circuit

Training needs thousands of energy evaluations. We train with a small NumPy statevector simulator that applies
exactly the same operations (phase by cost, XY trades on the ring, or RX for plain QAOA). We then **verify it
against Qiskit's `Statevector` of the real circuit** for random angles. Final answers are always sampled from the
Qiskit circuit (Section 6).

> **📘 Why this is legitimate.** The circuit is the specification; the fast simulator is only a speed-up for the
> optimizer loop (about 0.5 ms per evaluation instead of tens of ms). The check below shows they produce the same
> quantum state (overlap = 1). On hardware, each evaluation would instead be a batch of circuit runs.

In [ ]:
_idx = np.arange(N)
def _pair_index(i, j):
    sel = _idx[(((_idx >> i) & 1) == 1) & (((_idx >> j) & 1) == 0)]
    return sel, sel ^ (1 << i) ^ (1 << j)          # partner: stock i swapped for stock j
PAIRS = {e: _pair_index(*e) for e in EDGES_EVEN + EDGES_ODD}

def apply_xy_ring(psi, beta):
    c, s = np.cos(beta), -1j * np.sin(beta)
    for e in EDGES_EVEN + EDGES_ODD:
        i1, i2 = PAIRS[e]
        u, v = psi[i1].copy(), psi[i2].copy()
        psi[i1], psi[i2] = c * u + s * v, s * u + c * v
    return psi

def apply_rx_all(psi, beta):
    c, s = np.cos(beta), -1j * np.sin(beta)
    for i in range(n):
        t = psi.reshape(-1, 2, 2 ** i)
        u, v = t[:, 0, :].copy(), t[:, 1, :].copy()
        t[:, 0, :], t[:, 1, :] = c * u + s * v, s * u + c * v
    return psi

def simulate(theta, cnorm, mixer, psi0):
    p = len(theta) // 2
    psi = psi0.copy()
    for l in range(p):
        psi *= np.exp(-1j * theta[l] * cnorm)
        psi = apply_xy_ring(psi, theta[p + l]) if mixer == "xy" else apply_rx_all(psi, theta[p + l])
    return psi

PSI_DICKE = target.astype(complex)
PSI_PLUS = np.ones(N, complex) / np.sqrt(N)
PSI_BASIS = np.zeros(N, complex); PSI_BASIS[sum(1 << i for i in range(n - K, n))] = 1
CNORM_XY = (C0 - C_MIN_F) * SCALE_XY            # costs rescaled to [0, 1] on valid portfolios

def plain_setup(lam, mu_=None, sigma_=None):
    """Penalized costs for plain QAOA, divided by the valid-portfolio range (same γ scale as the XY run)."""
    mu_ = mu if mu_ is None else mu_
    sigma_ = sigma if sigma_ is None else sigma_
    c0, ct = cost_for(mu_, sigma_), cost_for(mu_, sigma_, lam)
    scale = 1 / (c0[FEAS].max() - c0[FEAS].min())
    return (ct - ct.min()) * scale, scale

# 5 random angle sets at each depth p = 1, 2, 3, for each circuit type
check_rng = np.random.default_rng(1)
rows = []
for mixer, init, lam in [("xy", "dicke", 0.0), ("xy", "basis", 0.0), ("x", "plus", 2 * LAMBDA_STAR)]:
    if mixer == "xy":
        cn, cs, psi0 = CNORM_XY, SCALE_XY, PSI_DICKE if init == "dicke" else PSI_BASIS
    else:
        (cn, cs), psi0 = plain_setup(lam), PSI_PLUS
    for p in (1, 2, 3):
        qc, g_, b_ = qaoa_circuit(mixer, p, lam, cs, init)
        for _ in range(5):
            theta = check_rng.uniform(-2, 2, 2 * p)
            sv = Statevector(bind(qc, g_, b_, theta)).data
            fast = simulate(theta, cn, mixer, psi0)
            rows.append({"circuit": f"{mixer} mixer, {init} start", "p": p,
                         "overlap": abs(np.vdot(fast, sv)) ** 2, "P(valid)": np.sum(np.abs(sv[FEAS]) ** 2)})
check_df = pd.DataFrame(rows)
for name, d in check_df.groupby("circuit", sort=False):
    print(f"{name:>20}: {len(d)} checks, min |⟨fast|Qiskit⟩|² = {d['overlap'].min():.10f}, "
          f"probability on valid portfolios {d['P(valid)'].min():.6f} – {d['P(valid)'].max():.6f}")
assert check_df["overlap"].min() > 1 - 1e-9

The XY rows show probability **exactly 1** on valid portfolios at random angles: the budget can never break.
The plain X mixer leaks most of its probability onto invalid portfolios.

## 6. Training the XY-mixer QAOA, depth p = 1 … 6

**Optimizer.** L-BFGS-B (a gradient-based method, gradients by finite differences) from 8 random starts plus
**two warm starts** built from the best angles at depth p−1: INTERP (Zhou et al. 2020; stretch the angle
schedule to one more layer by linear interpolation) and "repeat the last layer". We also report how many
of the runs converged.

> **📘 Which number does QAOA minimize?** The *average* cost $\langle C \rangle$ of its output. A low average
> means probability is concentrated on good portfolios, but it is not the same as maximizing the chance of the
> single best one.
>
> **📘 Metrics we report.**
> - $r = \dfrac{C_{\max} - \langle C\rangle}{C_{\max} - C_{\min}}$ over the **924 valid portfolios** (the only ones
>   this circuit can produce), so it is not flattered by fined invalid bitstrings. A uniformly random valid
>   portfolio scores `R_RANDOM` (printed in Section 1).
> - **P(best)**: chance one shot returns the optimal portfolio. Random guessing among valid portfolios gives 1/924 = 0.11%.
> - **P(top 10)**: chance one shot lands in the 10 best portfolios. Random gives 10/924 = 1.1%.
> - **Shots for 99% confidence**: runs needed to see the best portfolio at least once with 99% probability,
>   $\lceil \ln 0.01 / \ln(1 - P_\text{best}) \rceil$.

In [ ]:
TOP10 = order[:10]

def interp_init(x):
    """INTERP (Zhou et al. 2020): stretch a length-p angle schedule to length p+1."""
    p = len(x)
    xp = np.concatenate([[0.0], x, [0.0]])
    return np.array([((i - 1) / p) * xp[i - 1] + ((p - i + 1) / p) * xp[i] for i in range(1, p + 2)])

def warm_starts(theta):
    """Starts for depth p+1 from the best depth-p angles: INTERP and 'repeat the last layer' (deduplicated)."""
    p = len(theta) // 2
    g, b = theta[:p], theta[p:]
    a = np.concatenate([interp_init(g), interp_init(b)])
    r = np.concatenate([np.append(g, g[-1]), np.append(b, b[-1])])
    return [a] if np.allclose(a, r) else [a, r]

def train(energy, p, warm=(), n_starts=8, seed=0):
    """Best of L-BFGS-B runs from the warm starts plus n_starts random starts."""
    rs = np.random.default_rng(seed)
    starts = list(warm) + [rs.uniform(-np.pi, np.pi, 2 * p) for _ in range(n_starts)]
    best, evals, conv = None, 0, 0
    for x0 in starts:
        res = minimize(energy, x0, method="L-BFGS-B")
        evals += res.nfev
        conv += bool(res.success)
        if best is None or res.fun < best.fun:
            best = res
    return best.x, evals, f"{conv}/{len(starts)}"

def shots_99(p_best):
    return int(np.ceil(np.log(0.01) / np.log(1 - p_best))) if p_best > 1e-9 else np.inf

def xy_metrics(theta, psi0=PSI_DICKE):
    probs = np.abs(simulate(theta, CNORM_XY, "xy", psi0)) ** 2
    mean_cost = probs @ C0
    return {"⟨C⟩": mean_cost, "r": (C_MAX_F - mean_cost) / (C_MAX_F - C_MIN_F),
            "P(valid)": probs[FEAS].sum(), "P(best)": probs[BEST], "× random": probs[BEST] * len(FEAS),
            "P(top 10)": probs[TOP10].sum(), "shots for 99%": shots_99(probs[BEST])}

def depth_sweep(psi0, p_max=6, n_starts=8, verbose=True):
    rows, thetas, warm, t0 = [], {}, (), time.time()
    for p in range(1, p_max + 1):
        energy = lambda th: float(np.abs(simulate(th, CNORM_XY, "xy", psi0)) ** 2 @ CNORM_XY)
        theta, evals, conv = train(energy, p, warm=warm, n_starts=n_starts, seed=p)
        warm, thetas[p] = warm_starts(theta), theta
        rows.append({"p": p, **xy_metrics(theta, psi0), "training evaluations": evals, "converged starts": conv})
    if verbose:
        print(f"done in {time.time() - t0:.0f}s")
    return pd.DataFrame(rows).set_index("p"), thetas

xy_df, xy_thetas = depth_sweep(PSI_DICKE)
xy_df.style.format({"⟨C⟩": "{:.4f}", "r": "{:.3f}", "P(valid)": "{:.3f}", "P(best)": "{:.2%}", "× random": "{:.1f}×",
                    "P(top 10)": "{:.1%}", "shots for 99%": "{:.0f}", "training evaluations": "{:,}"})

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4))
ps = xy_df.index
ax1.plot(ps, xy_df["r"], "o-", color=BLUE)
for p, r in zip(ps, xy_df["r"]):
    ax1.annotate(f"{r:.3f}", (p, r), textcoords="offset points", xytext=(0, 9), ha="center", color=INK)
ax1.axhline(R_RANDOM, color=MUTED, lw=1, ls="--")
ax1.text(6, R_RANDOM + 0.01, f"random valid portfolio ({R_RANDOM:.2f})", ha="right", color=MUTED, fontsize=9)
ax1.set(xlabel="QAOA depth p", ylabel="approximation ratio r (valid set)", xticks=list(ps),
        title="Average answer quality vs depth", ylim=(0.4, 1.0))

ax2.plot(ps, xy_df["P(top 10)"], "o-", color=BLUE, label="lands in the top 10")
ax2.plot(ps, xy_df["P(best)"], "s-", color=ORANGE, label="the single best portfolio")
ax2.axhline(10 / len(FEAS), color=MUTED, lw=1, ls="--")
ax2.text(6, 10 / len(FEAS) + 0.012, "random: top 10 (1.1%)", ha="right", color=MUTED, fontsize=9)
ax2.set(xlabel="QAOA depth p", ylabel="probability per shot", xticks=list(ps), ylim=(0, 0.6),
        title="Chance of a top answer per shot")
ax2.legend(frameon=False, loc="upper left")
plt.tight_layout(); plt.show()

### Check with the real Qiskit circuit

We bind the trained angles at the deepest p into the Qiskit circuit, sample it with `StatevectorSampler`,
and compare against brute force.

In [ ]:
P_FINAL = max(xy_thetas)
theta_final = xy_thetas[P_FINAL]
qc_final, g_, b_ = qaoa_circuit("xy", P_FINAL, 0.0, SCALE_XY)
qc_final = bind(qc_final, g_, b_, theta_final)
qc_final.measure_all()
counts = StatevectorSampler(seed=7).run([qc_final], shots=8192).result()[0].data.meas.get_counts()

shots = sum(counts.values())
by_key = {int(key, 2): c for key, c in counts.items()}
p_valid = sum(c for k, c in by_key.items() if WEIGHT[k] == K) / shots
rank_of = {k: r + 1 for r, k in enumerate(order)}
top = sorted(by_key.items(), key=lambda kv: -kv[1])[:8]
common = set.intersection(*(set(stocks(k)) for k, _ in top))
print(f"p = {P_FINAL}, 8,192 shots: {p_valid:.1%} of shots are valid (exactly {K} stocks)")
print(f"Best portfolio {stocks(BEST)} sampled in {by_key.get(BEST, 0) / shots:.2%} of shots "
      f"(ideal {xy_df.loc[P_FINAL, 'P(best)']:.2%})")
print(f"Stocks held by all {len(top)} most-sampled portfolios: {sorted(common)}")
pd.DataFrame([{"stocks bought": stocks(k), "share of shots": f"{c / shots:.2%}", "true rank (of 924)": rank_of[k],
               "cost": round(float(C0[k]), 4)} for k, c in top])

## 7. Comparison 1: plain QAOA with a penalty (X mixer) at 12 qubits

To show the mixer is what makes the difference, we run the Beginner approach at the new size: start in
$|+\rangle^{\otimes 12}$, X mixer, penalty λ. We give it a fair chance by trying several λ (**1.05, 1.3, 2 and 4 × λ\***;
not exactly λ*, where the optimum is tied), the same optimizer, starts and warm starts, and the same γ scale as
the XY run (costs divided by the valid-portfolio range).

> **📘 Two ways to compare.** *P(best)* counts every shot, so invalid shots are wasted shots: this is what decides
> how many shots you need. *P(best | valid)* first throws away invalid shots (post-selection, free to check
> classically) and asks how good the remaining answers are. We report both.

In [ ]:
LAMBDA_FACTORS = [1.05, 1.3, 2.0, 4.0]       # multiples of λ*; not 1.0, where the optimum is tied

def plain_sweep(lam, p_max=4, n_starts=8):
    cn, _ = plain_setup(lam)
    rows, warm = [], ()
    for p in range(1, p_max + 1):
        energy = lambda th: float(np.abs(simulate(th, cn, "x", PSI_PLUS)) ** 2 @ cn)
        theta, evals, conv = train(energy, p, warm=warm, n_starts=n_starts, seed=p)
        warm = warm_starts(theta)
        probs = np.abs(simulate(theta, cn, "x", PSI_PLUS)) ** 2
        valid = probs[FEAS].sum()
        rows.append({"λ / λ*": lam / LAMBDA_STAR, "p": p, "P(valid)": valid, "P(best)": probs[BEST],
                     "P(best | valid)": probs[BEST] / valid, "P(top 10)": probs[TOP10].sum(),
                     "training evaluations": evals, "converged starts": conv})
    return rows

t0 = time.time()
plain_df = pd.DataFrame([row for f in LAMBDA_FACTORS for row in plain_sweep(f * LAMBDA_STAR)])
print(f"done in {time.time() - t0:.0f}s")
plain_df.style.format({"λ / λ*": "{:.2f}", "P(valid)": "{:.1%}", "P(best)": "{:.2%}", "P(best | valid)": "{:.2%}",
                       "P(top 10)": "{:.1%}", "training evaluations": "{:,}"}).hide(axis="index")

In [ ]:
best_plain = plain_df.loc[plain_df.groupby("p")["P(best)"].idxmax()].set_index("p")
best_plain_ps = plain_df.groupby("p")["P(best | valid)"].max()
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(xy_df.index[:4], xy_df["P(best)"].iloc[:4], "o-", color=BLUE, label="XY mixer + Dicke start (ours)")
ax.plot(best_plain.index, best_plain["P(best)"], "s-", color=ORANGE, label="plain QAOA, best λ at each p")
ax.plot(best_plain_ps.index, best_plain_ps.values, "s--", color=ORANGE, alpha=0.5,
        label="plain QAOA after post-selection, best λ")
ax.axhline(1 / len(FEAS), color=MUTED, lw=1, ls="--")
ax.text(4, 1 / len(FEAS) + 0.002, "random valid pick (0.11%)", ha="right", color=MUTED, fontsize=9)
ax.set_ylim(0, max(xy_df["P(best)"].iloc[:4].max(), best_plain_ps.max()) * 1.25)
ax.set(xlabel="QAOA depth p", ylabel="P(best portfolio) per shot", xticks=[1, 2, 3, 4],
       title="Same depth, same optimizer: chance of the best portfolio")
ax.legend(frameon=False, loc="upper left")
plt.tight_layout(); plt.show()

ratio_raw = xy_df["P(best)"].iloc[:4].values / best_plain["P(best)"].values
ratio_ps = xy_df["P(best)"].iloc[:4].values / best_plain_ps.values       # XY is always valid, so P(best | valid) = P(best)
print("XY / plain, every shot counted (p = 1..4):  ", ", ".join(f"{r:.1f}×" for r in ratio_raw))
print("XY / plain, after post-selection (p = 1..4):", ", ".join(f"{r:.1f}×" for r in ratio_ps))

### Is the gain specific to one dataset?

Everything above uses one dataset (seed 2026). We repeat the XY-vs-plain comparison (p = 1…4) on four more
datasets from the same generator and report three measures: **P(best)**, **P(top 10)** and **r** (for plain QAOA,
r is computed on its valid shots only, which flatters it). Plain QAOA tries the two λ multiples that gave the
highest average P(best) above, and we keep its best λ for each measure and depth.

To keep it short, each depth uses the warm starts plus **3** random starts (the main run uses 8). We tested this:
rerunning the dataset where XY did worst with 8 starts gave identical numbers (`RECHECK_WITH_8_STARTS` repeats
that check). We also print each dataset's **gap between its best and second-best portfolio**: when the top two
are nearly tied, P(best) alone is a harsh measure. Takes about 6–8 minutes; set `RUN_SEEDS = False` to skip.

In [ ]:
RUN_SEEDS = True
EXTRA_SEEDS = [1, 2, 3, 4]
P_SEED, STARTS_SEED = 4, 3
RECHECK_WITH_8_STARTS = False          # True: rerun the dataset where XY did worst with 8 random starts (~3 min)
seed_factors = [float(f) for f in plain_df.groupby("λ / λ*")["P(best)"].mean().nlargest(2).index]
MAIN_GAP = (C0[order[1]] - C0[BEST]) / (C_MAX_F - C_MIN_F)

def depth_curve(cnorm, c0, best, top10, mixer, psi0, n_starts):
    """P(best), P(top 10) and r on valid shots at p = 1..P_SEED, trained like the main run."""
    cmin, cmax = c0[FEAS].min(), c0[FEAS].max()
    out, warm = [], ()
    for p in range(1, P_SEED + 1):
        energy = lambda th: float(np.abs(simulate(th, cnorm, mixer, psi0)) ** 2 @ cnorm)
        theta, _, _ = train(energy, p, warm=warm, n_starts=n_starts, seed=p)
        warm = warm_starts(theta)
        pr = np.abs(simulate(theta, cnorm, mixer, psi0)) ** 2
        pv = pr[FEAS].sum()
        out.append({"P(best)": pr[best], "P(top 10)": pr[top10].sum(),
                    "r": (cmax - (pr[FEAS] @ c0[FEAS]) / pv) / (cmax - cmin)})
    return pd.DataFrame(out, index=range(1, P_SEED + 1))

def dataset_comparison(seed, n_starts):
    mu_s, sigma_s = make_data(seed)
    c0 = cost_for(mu_s, sigma_s)
    ranked = FEAS[np.argsort(c0[FEAS])]
    best_s, top10_s = ranked[0], ranked[:10]
    rng_s = c0[FEAS].max() - c0[FEAS].min()
    gap = (c0[ranked[1]] - c0[best_s]) / rng_s
    lam_s, _ = lambda_star(c0, best_s)
    xy = depth_curve((c0 - c0[FEAS].min()) / rng_s, c0, best_s, top10_s, "xy", PSI_DICKE, n_starts)
    plains = [depth_curve(plain_setup(f * lam_s, mu_s, sigma_s)[0], c0, best_s, top10_s, "x", PSI_PLUS, n_starts)
              for f in seed_factors]
    plain = pd.concat(plains).groupby(level=0).max()          # best λ for each measure and depth
    return xy, plain, gap

if RUN_SEEDS:
    t0, rows, seed_gaps = time.time(), [], {}
    for s in EXTRA_SEEDS:
        xy, plain, seed_gaps[s] = dataset_comparison(s, STARTS_SEED)
        for p in xy.index:
            rows.append({"dataset seed": s, "p": p, **{f"XY {k}": v for k, v in xy.loc[p].items()},
                         **{f"plain {k}": v for k, v in plain.loc[p].items()}})
    seed_df = pd.DataFrame(rows)
    for m in ["P(best)", "P(top 10)"]:
        seed_df[f"XY / plain, {m}"] = seed_df[f"XY {m}"] / seed_df[f"plain {m}"].clip(lower=1e-12)
    print(f"done in {time.time() - t0:.0f}s; plain QAOA used λ = {seed_factors} × λ* of each dataset")
    print("Gap best → 2nd best (share of cost range): " +
          ", ".join(f"seed {s}: {g:.2%}" for s, g in seed_gaps.items()) + f"; main dataset: {MAIN_GAP:.2%}")
    print("Main dataset (seed 2026), XY / plain P(best):", ", ".join(f"{r:.1f}×" for r in ratio_raw))
    for m in ["P(best)", "P(top 10)"]:
        print(f"\nXY / plain, {m} (rows: depth p; columns: dataset seed)")
        display(seed_df.pivot(index="p", columns="dataset seed", values=f"XY / plain, {m}").style.format("{:.1f}×"))
    print("\nr (plain: valid shots only)")
    display(seed_df.pivot(index="p", columns="dataset seed", values=["XY r", "plain r"]).style.format("{:.3f}"))

    if RECHECK_WITH_8_STARTS:
        worst = int(seed_df[seed_df.p == P_SEED].sort_values("XY / plain, P(best)").iloc[0]["dataset seed"])
        t0 = time.time()
        xy8, plain8, _ = dataset_comparison(worst, 8)
        print(f"\nRecheck of dataset {worst} with 8 random starts ({time.time() - t0:.0f}s):")
        for p in xy8.index:
            print(f"p={p}: XY P(best) {xy8.loc[p, 'P(best)']:.2%} | plain {plain8.loc[p, 'P(best)']:.2%}")
else:
    seed_df, seed_gaps = None, {}
    print("Skipped (RUN_SEEDS = False).")

## 8. Ablation: why the Dicke start matters

The Dicke preparation is the most expensive part of the circuit on hardware (Section 10). A cheaper option is
to start from **one** valid portfolio (just 6 X gates) and let the XY mixer explore from there.

> **📘 What to expect.** From a single bitstring, the first cost layer only adds a global phase, which changes
> nothing physical, so all exploration has to come from the mixer. And because the optimizer minimizes the
> *average* cost, it may prefer to stay on a decent starting portfolio rather than spread out.
>
> **📘 A fair test.** One starting portfolio could be lucky or unlucky, so we try five: the original (stocks 6–11)
> plus the portfolios ranked 100, 462 (the median), 800 and 924 (the worst). For each we report the chance of the
> best portfolio, the chance of **staying on the start**, the largest share on any single portfolio, and the
> pattern of the trained mixer angles β. At β = kπ an XY gate only adds phases (the portfolio stays); at
> β = π/2 it is a full swap (the portfolio is moved, but stays a single portfolio). Only other angles spread
> probability over many portfolios. We also show r next to the Dicke run's r, because r is what the optimizer
> actually minimizes.

In [ ]:
start_key = sum(1 << i for i in range(n - K, n))
rank_of = {k: r + 1 for r, k in enumerate(order)}
basis_starts = {f"stocks 6–11 (rank {rank_of[start_key]})": start_key}
for rk in (100, 462, 800, 924):
    basis_starts[f"rank {rk}"] = order[rk - 1]

t0, rows, basis_thetas = time.time(), [], {}
for label, k in basis_starts.items():
    psi0 = np.zeros(N, complex); psi0[k] = 1
    df_, th_ = depth_sweep(psi0, p_max=4, verbose=False)
    basis_thetas[label] = th_
    for p in df_.index:
        betas = th_[p][p:]
        probs = np.abs(simulate(th_[p], CNORM_XY, "xy", psi0)) ** 2
        d_pi = np.abs((betas + np.pi / 2) % np.pi - np.pi / 2)        # distance to 0, π, 2π, …
        d_half = np.abs(betas % np.pi - np.pi / 2)                    # distance to π/2, 3π/2, …
        if d_pi.max() < 0.05:
            pattern = "stays (β ≈ kπ)"
        elif d_half.max() < 0.05:
            pattern = "full swaps (β ≈ π/2)"
        elif np.minimum(d_pi, d_half).max() < 0.05:
            pattern = "stay/swap layers"
        else:
            pattern = "mixes"
        rows.append({"start": label, "p": p, "r": df_.loc[p, "r"], "Dicke r": xy_df.loc[p, "r"],
                     "P(best)": df_.loc[p, "P(best)"], "P(top 10)": df_.loc[p, "P(top 10)"],
                     "P(stay on start)": probs[k], "largest single-portfolio share": probs.max(),
                     "β pattern": pattern})
basis_df = pd.DataFrame(rows)
print(f"done in {time.time() - t0:.0f}s")
basis_df.style.format({"r": "{:.3f}", "Dicke r": "{:.3f}", "P(best)": "{:.2%}", "P(top 10)": "{:.1%}",
                       "P(stay on start)": "{:.1%}", "largest single-portfolio share": "{:.1%}"}).hide(axis="index")

## 9. Comparison 2: classical methods

| Method | Idea |
|---|---|
| **Brute force** | Score all 924 portfolios. Exact; impossible at large n. |
| **Greedy** | Add stocks one at a time, each time picking the one that lowers cost most. |
| **Greedy + swap** | Then keep swapping one held stock for one unheld stock while cost improves (local search). |
| **Simulated annealing (SA)** | Random swaps; accept worse swaps with a probability that shrinks over time, to escape local minima. Repeated 300 times to measure a success rate. |
| **Random sampling** | Draw valid portfolios at random. |

> **📘 Cost evaluations.** For classical methods we count how many portfolios each method scores. QAOA's
> equivalent is the number of circuit evaluations (each needing many shots on hardware) used in training, plus
> the shots used to read out the answer.

In [ ]:
def key_of(S):
    return sum(1 << i for i in S)

def greedy():
    S, evals = [], 0
    for _ in range(K):
        cands = [(C0[key_of(S + [i])], i) for i in range(n) if i not in S]
        evals += len(cands)
        S.append(min(cands)[1])
    return key_of(S), evals

def greedy_swap():
    k, evals = greedy()
    S, improved = list(stocks(k)), True
    while improved:
        improved = False
        for out_ in list(S):
            for in_ in [i for i in range(n) if i not in S]:
                T = [i for i in S if i != out_] + [in_]
                evals += 1
                if C0[key_of(T)] < C0[key_of(S)] - 1e-12:
                    S, improved = T, True
                    break
            if improved:
                break
    return key_of(S), evals

def annealing(rs, steps, T0=0.05, T1=0.001):
    S = list(rs.choice(n, K, replace=False))
    cur = C0[key_of(S)]
    best = (cur, key_of(S))
    for t in range(steps):
        T = T0 * (T1 / T0) ** (t / steps)
        out_ = rs.choice(S); in_ = rs.choice([i for i in range(n) if i not in S])
        T_ = [i for i in S if i != out_] + [in_]
        c = C0[key_of(T_)]
        if c < cur or rs.random() < np.exp((cur - c) / T):
            S, cur = T_, c
        best = min(best, (cur, key_of(S)))
    return best[1], steps + 1

rs = np.random.default_rng(0)
rows = [{"method": "brute force", "finds the best?": "always", "cost evaluations": len(FEAS)}]
for name, fn in [("greedy", greedy), ("greedy + swap", greedy_swap)]:
    k, ev = fn()
    rows.append({"method": name, "finds the best?": "yes" if k == BEST else f"no (rank {rank_of[k]})",
                 "cost evaluations": ev})
for steps in [50, 100, 200, 400]:
    hits = np.mean([annealing(rs, steps)[0] == BEST for _ in range(300)])
    rows.append({"method": f"simulated annealing, {steps} steps", "finds the best?": f"{hits:.0%} of runs",
                 "cost evaluations": steps + 1})
for draws in [100, 500]:
    hit = 1 - (1 - 1 / len(FEAS)) ** draws
    rows.append({"method": f"random sampling, {draws} draws", "finds the best?": f"{hit:.0%} of runs",
                 "cost evaluations": draws})
classical_df = pd.DataFrame(rows).set_index("method")
classical_df

## 10. Hardware awareness: what these circuits cost on a real IBM chip

We transpile for **FakeSherbrooke** (a local copy of a 127-qubit IBM Eagle device: its heavy-hex wiring,
native gates and error rates) and count two-qubit ECR gates, the main source of error. The result changes with
the transpiler seed and the optimization level, so we try **5 seeds at levels 2 and 3** (level 3 is the setting
of our Advanced notebook; level 2 sometimes gives fewer gates). The table shows the median for each level and
the **best circuit overall**, which is the one we would run and the one used in the noisy simulation. We also transpile each circuit with **all-to-all wiring**
(native gates, no routing) to separate the cost of gate synthesis from the cost of routing.

> **📘 Why the counts explode.** Our cost has a term for **every pair** of stocks (66 pairs), but each physical
> qubit talks to only 2–3 neighbours. The transpiler inserts SWAPs to bring qubits together, and each SWAP costs
> 3 two-qubit gates. Each controlled-controlled rotation in the Dicke circuit also becomes several two-qubit gates.
>
> **📘 A quick fidelity estimate.** If each two-qubit gate fails with probability ε, a circuit with G of them
> succeeds roughly with probability $(1-\varepsilon)^G$, using the device's **median** ECR error and the best circuit's gate count. It ignores
> single-qubit, readout and idle errors, so it is **optimistic**.

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
fake = FakeSherbrooke()
ecr_err = float(np.median([v.error for v in fake.target["ecr"].values() if v is not None and v.error is not None]))
OPT_LEVEL, OPT_LEVELS, T_SEEDS = 3, (2, 3), [1, 2, 3, 4, 5]
LAMBDA_HW = 1.05 * LAMBDA_STAR
pms = {(lv, s): generate_preset_pass_manager(optimization_level=lv, backend=fake, seed_transpiler=s)
       for lv in OPT_LEVELS for s in T_SEEDS}
try:   # native gates, all-to-all wiring: shows the cost before routing
    pm_unrouted = generate_preset_pass_manager(optimization_level=OPT_LEVEL, basis_gates=["ecr", "rz", "sx", "x"],
                                               seed_transpiler=1)
except Exception as err:
    pm_unrouted = None
    print("Unrouted count skipped:", err)

def cost_layer(gamma, lam=0.0, cscale=SCALE_XY):
    const, a, b = spin_coeffs(lam)
    qc = QuantumCircuit(n)
    for i in range(n):
        qc.rz(2 * a[i] * cscale * gamma, i)
    for i in range(n):
        for j in range(i + 1, n):
            qc.rzz(2 * b[i, j] * cscale * gamma, i, j)
    return qc

def xy_mixer(beta):
    qc = QuantumCircuit(n)
    for e in EDGES_EVEN + EDGES_ODD:
        qc.append(XXPlusYYGate(2 * beta), list(e))
    return qc

def ecr(tc):
    return int(tc.count_ops().get("ecr", 0))

def hw_row(label, circ):
    """Transpile at each level and seed; return a summary row and the best circuit (with measurements)."""
    c = circ.copy(); c.measure_all()
    tcs = {key: pm.run(c) for key, pm in pms.items()}
    g = {key: ecr(tc) for key, tc in tcs.items()}
    k_best = min(g, key=g.get)
    unrouted = np.nan
    if pm_unrouted is not None:
        try:
            unrouted = ecr(pm_unrouted.run(circ))
        except Exception:
            pass
    row = {"circuit": label}
    for lv in OPT_LEVELS:
        row[f"ECR median, level {lv}"] = int(np.median([g[(lv, s)] for s in T_SEEDS]))
    row.update({"ECR best": g[k_best], "best setting": f"level {k_best[0]}, seed {k_best[1]}",
                "ECR without routing": unrouted, "depth (best)": tcs[k_best].depth(),
                "est. success (1-ε)^best": (1 - ecr_err) ** g[k_best]})
    return row, tcs[k_best]

g1, b1 = xy_thetas[1]
hw_circuits = [
    ("Dicke preparation only", dicke_circuit(n, K)),
    ("cost layer only (66 RZZ)", cost_layer(g1)),
    ("XY mixer only (12 XY)", xy_mixer(b1)),
    ("XY QAOA, p = 1", bind(*qaoa_circuit("xy", 1, 0.0, SCALE_XY), xy_thetas[1])),
    ("XY QAOA, p = 2", bind(*qaoa_circuit("xy", 2, 0.0, SCALE_XY), xy_thetas[2])),
    ("plain QAOA, p = 1", bind(*qaoa_circuit("x", 1, LAMBDA_HW, SCALE_XY), np.array([0.5, 0.3]))),
    ("plain QAOA, p = 2", bind(*qaoa_circuit("x", 2, LAMBDA_HW, SCALE_XY), np.array([0.5, 0.5, 0.3, 0.3]))),
]
t0, hw_rows, transpiled = time.time(), [], {}
for label, circ in hw_circuits:
    row, tc = hw_row(label, circ)
    hw_rows.append(row); transpiled[label] = tc
hw_df = pd.DataFrame(hw_rows).set_index("circuit")
dicke_share = hw_df.loc["Dicke preparation only", "ECR best"] / hw_df.loc["XY QAOA, p = 1", "ECR best"]
print(f"Median ECR error on FakeSherbrooke: {ecr_err:.2%}; transpiled {len(hw_circuits)} circuits × "
      f"{len(pms)} settings in {time.time() - t0:.0f}s")
print(f"Dicke preparation alone is {dicke_share:.0%} of the p = 1 XY circuit's two-qubit gates (best circuits; "
      f"blocks transpiled separately, so the parts need not add up exactly)")
hw_df.style.format({"ECR without routing": "{:.0f}", "est. success (1-ε)^best": "{:.2%}"})

### Noisy simulation (free) and a simple error mitigation

We run the p = 1 XY circuit (the best of the 5 transpiler seeds) through FakeSherbrooke's noise model. Noise
breaks the "always 6 stocks" guarantee, so we apply **post-selection**: discard every shot that does not buy
exactly 6 stocks.

> **📘 Post-selection** is a cheap error-mitigation trick that works *because* of our mixer. A noise-free run
> can only produce valid portfolios, so any invalid shot is a detected error. It removes detectable errors but
> not errors that happen to land on another valid portfolio.
>
> **📘 Error bars.** With a few hundred shots, one lucky shot moves a rate by almost 1%. We show 95% confidence
> intervals (Wilson) next to every rate. When the circuit touches at most 13 qubits, the simulator computes the
> full noisy state once (density-matrix method), so extra shots are almost free and we take **100,000**.
> Otherwise it simulates shot by shot, which is much slower, and we take 4,096. The method, shots and run time
> are printed.

In [ ]:
tc_noisy = transpiled["XY QAOA, p = 1"]          # the best of the 10 transpiled versions
active = {tc_noisy.find_bit(qb).index for inst in tc_noisy.data if inst.operation.name != "barrier" for qb in inst.qubits}
method = "density_matrix" if len(active) <= 13 else "automatic"
NOISY_SHOTS = 100_000 if method == "density_matrix" else 4096   # density matrix: state computed once, shots are cheap

def run_noisy(method_, shots_):
    sim = AerSimulator.from_backend(fake, method=method_, seed_simulator=7)
    return sim.run(tc_noisy, shots=shots_).result()

t0 = time.time()
try:
    noisy_counts = run_noisy(method, NOISY_SHOTS).get_counts()
except Exception as err:                      # e.g. not enough memory for the density matrix
    print(f"'{method}' failed ({err}); falling back to shot-by-shot simulation with 4,096 shots")
    method, NOISY_SHOTS = "automatic", 4096
    noisy_counts = run_noisy(method, NOISY_SHOTS).get_counts()
print(f"{len(active)} active qubits → method '{method}'; {NOISY_SHOTS:,} shots took {time.time() - t0:.0f}s")

def wilson(k, m, z=1.96):
    """95% confidence interval for a rate of k hits in m shots."""
    if m == 0:
        return np.nan, np.nan
    ph, d = k / m, 1 + z ** 2 / m
    c = (ph + z ** 2 / (2 * m)) / d
    h = z * np.sqrt(ph * (1 - ph) / m + z ** 2 / (4 * m * m)) / d
    return max(0.0, c - h), min(1.0, c + h)

def readout(counts, postselect=False):
    """Rates per shot (or per kept shot) and the raw hit counts behind them."""
    by_key = {int(key, 2): c for key, c in counts.items()}
    total = sum(by_key.values())
    valid = {k: c for k, c in by_key.items() if WEIGHT[k] == K}
    nv = sum(valid.values())
    base = nv if postselect else total
    hits = {"valid": nv, "best": valid.get(BEST, 0), "top 10": sum(valid.get(k, 0) for k in TOP10), "shots used": base}
    vals = {"shots kept": base / total, "P(valid)": nv / base, "P(best)": hits["best"] / base,
            "P(top 10)": hits["top 10"] / base}
    return vals, hits

NOISY = {"noisy": readout(noisy_counts), "noisy + post-selection": readout(noisy_counts, postselect=True)}
ideal = {"shots kept": 1.0, **{k: xy_df.loc[1, k] for k in ["P(valid)", "P(best)", "P(top 10)"]}}
noise_df = pd.DataFrame({"ideal (noiseless)": ideal, **{k: v[0] for k, v in NOISY.items()},
                         "random valid pick": {"shots kept": 1.0, "P(valid)": 1.0, "P(best)": 1 / len(FEAS),
                                               "P(top 10)": 10 / len(FEAS)}}).T
ci_df = pd.DataFrame({name: {label: f"{h[key]} of {h['shots used']} → {wilson(h[key], h['shots used'])[0]:.2%} – "
                                    f"{wilson(h[key], h['shots used'])[1]:.2%}"
                             for label, key in [("P(valid)", "valid"), ("P(best)", "best"), ("P(top 10)", "top 10")]}
                      for name, (_, h) in NOISY.items()}).T
display(noise_df.style.format("{:.2%}"))
print("Hits and 95% confidence intervals:")
display(ci_df)

In [ ]:
conds = ["ideal (noiseless)", "noisy", "noisy + post-selection", "random valid pick"]
vals = [noise_df.loc[c, "P(top 10)"] for c in conds]
err_lo, err_hi = [], []
for c, v in zip(conds, vals):
    if c in NOISY:
        h = NOISY[c][1]
        lo, hi = wilson(h["top 10"], h["shots used"])
        err_lo.append(v - lo); err_hi.append(hi - v)
    else:
        err_lo.append(0.0); err_hi.append(0.0)
fig, ax = plt.subplots(figsize=(8.5, 4.2))
bars = ax.bar(range(len(conds)), vals, 0.6, color=[BLUE, ORANGE, AQUA, "#b9b8b2"],
              yerr=[err_lo, err_hi], capsize=6, ecolor=INK)
for bar, v, e in zip(bars, vals, err_hi):
    ax.annotate(f"{v:.1%}", (bar.get_x() + bar.get_width() / 2, v + e), textcoords="offset points",
                xytext=(0, 4), ha="center", color=INK)
ax.set_xticks(range(len(conds)), ["ideal\n(noiseless)", "noisy", "noisy +\npost-selection", "random\nvalid pick"])
ax.set(ylabel="share of shots in the top 10 portfolios", ylim=(0, max(v + e for v, e in zip(vals, err_hi)) * 1.25),
       title=f"XY QAOA p = 1 under FakeSherbrooke noise ({NOISY_SHOTS:,} shots, 95% intervals)")
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()
print(f"Noisy run: {noise_df.loc['noisy', 'P(valid)']:.1%} of shots valid "
      f"(pure noise would give {len(FEAS) / N:.1%}); post-selection keeps {noise_df.loc['noisy + post-selection', 'shots kept']:.1%}")

## 11. Optional: one run on a real IBM QPU

Off by default (free plan: 10 QPU minutes per 28 days). It sends **one** sampling job of the p = 1 circuit
with simulator-trained angles, with dynamical decoupling and gate twirling. Given the gate counts above,
expect a mostly noisy result; post-selection is applied automatically.

In [ ]:
RUN_ON_HARDWARE = False

if RUN_ON_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
    service = QiskitRuntimeService()
    backend = service.least_busy(operational=True, simulator=False)
    circ = bind(*qaoa_circuit("xy", 1, 0.0, SCALE_XY), xy_thetas[1]); circ.measure_all()
    hw_circ = generate_preset_pass_manager(optimization_level=OPT_LEVEL, backend=backend, seed_transpiler=1).run(circ)
    print(backend.name, "| depth", hw_circ.depth(), "| 2q gates", hw_circ.num_nonlocal_gates())
    hw_sampler = SamplerV2(mode=backend)
    hw_sampler.options.dynamical_decoupling.enable = True
    hw_sampler.options.twirling.enable_gates = True
    job = hw_sampler.run([hw_circ], shots=4096)
    print("job id:", job.job_id())
    hw_counts = job.result()[0].data.meas.get_counts()
    noise_df.loc[f"hardware ({backend.name})"] = readout(hw_counts)[0]
    noise_df.loc["hardware + post-selection"] = readout(hw_counts, postselect=True)[0]
    display(noise_df.style.format("{:.2%}"))
else:
    print("Hardware run skipped (RUN_ON_HARDWARE = False).")

## 12. Results summary

In [ ]:
def ci_text(h, key):
    lo, hi = wilson(h[key], h["shots used"])
    return f"{h[key] / h['shots used']:.1%} ({h[key]} of {h['shots used']}; 95% CI {lo:.1%}–{hi:.1%})"

print("— Problem —")
print(f"Exact optimum: stocks {stocks(BEST)}, cost {C0[BEST]:.4f}; λ* = {LAMBDA_STAR:.4f}; "
      f"random valid portfolio r = {R_RANDOM:.3f}")
print(f"Random valid pick: P(best) = {1 / len(FEAS):.2%}, P(top 10) = {10 / len(FEAS):.1%}, "
      f"draws for 99% = {shots_99(1 / len(FEAS)):,}")

print("\n— XY QAOA (Dicke start) —")
for p, row in xy_df.iterrows():
    print(f"p={p}: P(best) {row['P(best)']:.2%} ({row['× random']:.0f}× random), top 10 {row['P(top 10)']:.1%}, "
          f"r {row['r']:.3f}, shots for 99% {row['shots for 99%']}, training evals {row['training evaluations']:,}, "
          f"converged {row['converged starts']}")
print(f"Qiskit sampling at p={P_FINAL}: {p_valid:.1%} valid, optimum in {by_key.get(BEST, 0) / shots:.2%} of "
      f"{shots:,} shots; favourite portfolio rank {rank_of[top[0][0]]} ({top[0][1] / shots:.1%}); "
      f"stocks in all top-{len(top)}: {sorted(common)}")

print("\n— Plain QAOA (X mixer + penalty) —")
print("Best P(best) per p:          ", ", ".join(f"p={p}: {v:.2%}" for p, v in best_plain["P(best)"].items()))
print("XY / plain, every shot:      ", ", ".join(f"{r:.1f}×" for r in ratio_raw))
print("XY / plain, post-selected:   ", ", ".join(f"{r:.1f}×" for r in ratio_ps))
for f, d in plain_df.groupby("λ / λ*"):
    print(f"λ = {f:.2f} λ*: P(valid) {d['P(valid)'].min():.0%}–{d['P(valid)'].max():.0%}, "
          f"P(best) {d['P(best)'].min():.2%}–{d['P(best)'].max():.2%}")
if seed_df is not None:
    print("\n— Other datasets (p = 1..4, XY / plain, median and min–max over 4 datasets) —")
    for m in ["P(best)", "P(top 10)"]:
        print(f"{m}: " + "; ".join(f"p={p}: {d.median():.1f}× ({d.min():.1f}–{d.max():.1f})"
                                   for p, d in seed_df.groupby("p")[f"XY / plain, {m}"]))
    print("r at p=4, XY vs plain (valid shots): " + ", ".join(
        f"seed {int(r_['dataset seed'])}: {r_['XY r']:.3f} vs {r_['plain r']:.3f}"
        for _, r_ in seed_df[seed_df.p == P_SEED].iterrows()))
    print("Best → 2nd-best gap: " + ", ".join(f"seed {s}: {g:.2%}" for s, g in seed_gaps.items())
          + f"; main dataset: {MAIN_GAP:.2%}")

print("\n— Single-portfolio starts —")
for label, d in basis_df.groupby("start", sort=False):
    last = d.iloc[-1]
    print(f"{label}: max P(best) {d['P(best)'].max():.2%}; at p=4: top 10 {last['P(top 10)']:.1%}, "
          f"P(stay) {last['P(stay on start)']:.0%}, largest single share {last['largest single-portfolio share']:.0%}, "
          f"r {last['r']:.3f} (Dicke {last['Dicke r']:.3f}), β: {last['β pattern']}")

print("\n— Classical —")
print(f"Greedy: {classical_df.loc['greedy', 'finds the best?']} with {classical_df.loc['greedy', 'cost evaluations']} "
      f"evaluations; greedy + swap {classical_df.loc['greedy + swap', 'cost evaluations']}; "
      f"SA 200 steps {classical_df.loc['simulated annealing, 200 steps', 'finds the best?']}")

print("\n— Hardware (FakeSherbrooke; best of 5 seeds × levels 2 and 3) —")
for label, row in hw_df.iterrows():
    print(f"{label}: {row['ECR best']:.0f} ECR ({row['best setting']}); medians: level 2 "
          f"{row['ECR median, level 2']:.0f}, level 3 {row['ECR median, level 3']:.0f}; without routing "
          f"{row['ECR without routing']:.0f}; est. success {row['est. success (1-ε)^best']:.2%}")
print(f"Dicke share of the p=1 XY circuit: {dicke_share:.0%}; XY p=1 / plain p=1 gate ratio: "
      f"{hw_df.loc['XY QAOA, p = 1', 'ECR best'] / hw_df.loc['plain QAOA, p = 1', 'ECR best']:.1f}×")

print(f"\n— Noisy simulation ({NOISY_SHOTS:,} shots, method '{method}') —")
h_all, h_ps = NOISY["noisy"][1], NOISY["noisy + post-selection"][1]
print(f"Valid shots: {ci_text(h_all, 'valid')}; pure noise would give {len(FEAS) / N:.1%}")
print(f"After post-selection: top 10 {ci_text(h_ps, 'top 10')}, best {ci_text(h_ps, 'best')}; "
      f"random: top 10 {10 / len(FEAS):.1%}, best {1 / len(FEAS):.2%}")

> ⚠️ **To refresh after the final run.** The numbers below are from the previous run. Update every number from
> the printed summary above (training evaluations, noisy valid share and the new results: P(best | valid),
> other datasets, the five basis starts, gate breakdown and confidence intervals).

### What we found
_Numbers from our run (all seeds fixed, so a rerun reproduces them; the noisy simulation uses 512 shots)._

1. **Everything is verified exactly.** The Dicke circuit matches the ideal state to 10⁻¹⁷, the spin-form cost matches
   brute force on all 4,096 bitstrings, the training simulator gives the same state as the Qiskit circuit (overlap 1.0),
   and the XY circuit keeps **100% of probability on 6-stock portfolios** at any angles.
2. **The budget-keeping QAOA works, with no penalty to tune.** The chance of sampling the single best portfolio
   (stocks 1, 5, 6, 8, 10, 11) rises from **1.2% at p = 1 to 6.1% at p = 6**, i.e. 11× to 57× better than a random
   valid pick (0.11%). The chance of landing in the top 10 rises from 13% to **47%** (random: 1.1%), and r rises
   from 0.695 to 0.841 (a random valid portfolio scores 0.46). Sampling the real Qiskit circuit confirms it: 100% of
   8,192 shots were valid and 6.3% were the optimum (ideal 6.1%).
3. **QAOA finds the right neighbourhood more than the exact answer.** At p = 6 its favourite portfolio is rank 10
   (16% of shots), and all 8 most-sampled portfolios contain stock 8 (7 of 8 contain stock 11). This follows from
   minimizing the *average* cost: it concentrates on a cluster of good portfolios.
4. **The mixer is what makes the difference.** With the same optimizer and depth, plain QAOA (X mixer + penalty,
   best of four λ values) samples the optimum only 0.08–1.5% of the time: **3.5× to 15× worse**. It also wastes
   20–50% of shots on invalid portfolios unless λ ≥ 4λ*, and at that λ its P(best) falls back to random (0.12%),
   the same trade-off we found at 6 stocks.
5. **The superposition start is essential.** Starting from a single portfolio (rank 46 of 924) leaves QAOA stuck
   there at every depth: P(best) = 0%. The expensive Dicke preparation is doing real work.
6. **Classical methods win at this size, and we say so plainly.** Greedy finds the optimum with 57 cost
   evaluations; simulated annealing succeeds 95% of the time with 200. QAOA used 369 (p = 1) to 28,028 (p = 6)
   training evaluations, each of which would be a batch of shots on hardware. Where QAOA does help: every answer is
   valid by construction, and it needs **73 shots** (p = 6) to see the optimum with 99% confidence versus about
   **4,250 random draws**.
7. **Hardware is the bottleneck.** On FakeSherbrooke's layout the Dicke preparation alone needs 532 two-qubit ECR
   gates, and the p = 1 circuit 795 (estimated success ≈ 0.2% at the median 0.78% gate error). In the noisy simulation
   only 23.6% of shots were valid, close to the 22.6% that pure noise gives, and after post-selection the top-10 rate
   (0.8%) was no better than random (1.1%). The plain circuit is 3.6× smaller (221 ECR gates), but its noiseless
   output is already near random, so neither version is hardware-ready at 12 stocks.

**Next steps (toward Advanced):** cut two-qubit gates (SWAP-network ordering of the 66 pair terms, a cheaper Dicke
preparation), add error suppression, then scale to 20+ stocks with a sector rule.

## 13. 📘 Study notes: concepts and likely judge questions

**Core concepts in one line each**

| Concept | One-line meaning |
|---|---|
| QUBO | A cost written with binary variables and at most pairwise products, no side constraints |
| Penalty term | Turns a rule into a cost; needs a strength λ above the threshold λ* |
| Hamming weight | Number of 1s in a bitstring; here, the number of stocks bought |
| Feasible subspace | All states that obey the rule; here, the 924 six-stock portfolios |
| Cost layer $e^{-i\gamma H_C}$ | Gives each portfolio a phase proportional to its cost; never changes which portfolio |
| X mixer | Flips stocks in and out independently; explores everything, including invalid portfolios |
| XY mixer | Trades one stock for another; explores only valid portfolios |
| Dicke state | Equal superposition of all bitstrings with exactly K ones |
| Approximation ratio r | How close the *average* answer is to the best, on a 0–1 scale |
| P(best) | Chance a single shot returns the optimal portfolio |
| Transpilation / routing | Rewriting a circuit into the chip's native gates and wiring, adding SWAPs |
| Post-selection | Discarding shots that break a rule the ideal circuit never breaks |

**Questions judges are likely to ask**

1. *Why did you pick the XY mixer route?*
   Our Beginner study showed the penalty needs careful tuning and a large λ destroys answer quality. The XY mixer
   removes the penalty entirely: every shot is valid by construction, so QAOA spends all its effort on ranking
   valid portfolios.
2. *How do you know every answer has exactly 6 stocks?*
   The Dicke start contains only 6-stock states, the cost layer only adds phases, and each XY gate only swaps
   $|01\rangle \leftrightarrow |10\rangle$. We verified it numerically: probability on valid portfolios = 1.
3. *Why not start from a single portfolio?*
   We tested it (Section 8): the optimizer stays on the starting portfolio. A cost layer acting on one bitstring
   is only a global phase, and minimizing the average cost rewards staying on a decent start.
4. *Is QAOA better than classical methods here?*
   No. Greedy finds the optimum with a few dozen evaluations. At 12 stocks this is a correctness and
   scaling study, not a speed claim. QAOA's gain is relative to random guessing and to plain QAOA.
5. *What would it take to run this well on hardware?*
   Fewer two-qubit gates: a cheaper Dicke preparation, cost terms ordered to match the chip's wiring (SWAP
   networks), error suppression (dynamical decoupling, twirling) and mitigation such as post-selection.
6. *Why train on a NumPy simulator?*
   Speed only. It applies the same operations as the Qiskit circuit, and we checked the two give the same state
   (overlap = 1). Final answers are sampled from the Qiskit circuit.

## Credits and tools used

- **Tutorial:** B. Tirado, *Combinatorial Optimization with QAOA: Max-Cut*, BasQ Qiskit Fall Fest 2026 (text CC BY 4.0, code Apache 2.0).
- **Challenge:** Qiskit Fall Fest 2026 IIIT-Delhi Track 2, adapted from the BasQ prompt by A. Rodriguez, B. Tirado and M. A. Jimenez.
- **Papers:**
  E. Farhi, J. Goldstone, S. Gutmann, *A Quantum Approximate Optimization Algorithm*, arXiv:1411.4028 (2014);
  S. Hadfield et al., *From the Quantum Approximate Optimization Algorithm to a Quantum Alternating Operator Ansatz*, Algorithms 12, 34 (2019) (constraint-preserving mixers);
  A. Bärtschi, S. Eidenbenz, *Deterministic Preparation of Dicke States*, FCT 2019, arXiv:1904.07358;
  L. Zhou et al., PRX 10, 021067 (2020) (layer-by-layer angle initialization).
- **Software:** Qiskit, Qiskit Aer, Qiskit IBM Runtime, NumPy, SciPy, pandas, Matplotlib.
- **AI assistance:** Claude (Anthropic) was used to help draft and debug code and explanations. All results were run and checked by the team.